In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# JOL Sub-metrics BenchmarkIndividual JOL metrics: gamma, ECE, recall.**Cognitive Science**: Nelson & Dunlosky (1991)

In [ ]:
"""JOL (Judgment-of-Learning) question/stimulus dataset.Novel association pairs that CANNOT be in training data.Each pair consists of an invented word and a definition,or a nonsense rule system. The model must:1. Study the associations2. Rate confidence of future recall (JOL)3. Perform distractor tasks4. Be tested on recallCategories:- WORD_DEF: Invented word → definition mappings- RULE: Novel rule systems (e.g., "In Zaplang, X means Y")- SEQUENCE: Novel pattern sequences to memorizeStimuli are procedurally varied across runs to prevent memorization."""import randomimport hashlibdef _seeded_word(seed: str, length: int = 6) -> str:    """Generate a pronounceable pseudoword from a seed."""    h = hashlib.sha256(seed.encode()).hexdigest()    consonants = "bcdfghjklmnprstvwz"    vowels = "aeiou"    word = ""    for i in range(length):        idx = int(h[i * 2:i * 2 + 2], 16)        if i % 2 == 0:            word += consonants[idx % len(consonants)]        else:            word += vowels[idx % len(vowels)]    return word.capitalize()# Fixed stimuli for reproducibility (but designed to be novel)JOL_WORD_PAIRS = [    # Easy pairs (concrete, imageable)    {"word": "Brelkano", "definition": "a small wooden bridge over a stream",     "difficulty": 1, "imageability": "high"},    {"word": "Tunnefex", "definition": "the sound of rain hitting a tin roof",     "difficulty": 1, "imageability": "high"},    {"word": "Glopwren", "definition": "a bird that only sings at dawn",     "difficulty": 1, "imageability": "high"},    {"word": "Verdashi", "definition": "a green gemstone found only in caves",     "difficulty": 1, "imageability": "high"},    {"word": "Plonkrit", "definition": "a heavy clay pot used for storing grain",     "difficulty": 1, "imageability": "high"},    # Medium pairs (semi-abstract)    {"word": "Feltromi", "definition": "the feeling of recognizing a place you've never been",     "difficulty": 2, "imageability": "medium"},    {"word": "Drasquil", "definition": "the moment just before understanding clicks",     "difficulty": 2, "imageability": "medium"},    {"word": "Wenvotch", "definition": "a tradition of leaving the last bite of food",     "difficulty": 2, "imageability": "medium"},    {"word": "Kelmapho", "definition": "the skill of navigating by starlight alone",     "difficulty": 2, "imageability": "medium"},    {"word": "Crinjota", "definition": "a pattern of cracks in dried mud",     "difficulty": 2, "imageability": "medium"},    # Hard pairs (abstract, low imageability)    {"word": "Phaxendu", "definition": "the property of being simultaneously necessary and impossible",     "difficulty": 3, "imageability": "low"},    {"word": "Zorblint", "definition": "a mathematical operator that reverses parity while preserving magnitude",     "difficulty": 3, "imageability": "low"},    {"word": "Quellmaf", "definition": "the tendency of systems to resist their optimal configuration",     "difficulty": 3, "imageability": "low"},    {"word": "Narvexti", "definition": "a logical relationship where A implies B only when C is unknown",     "difficulty": 3, "imageability": "low"},    {"word": "Blekthor", "definition": "the ratio of perceived complexity to actual information content",     "difficulty": 3, "imageability": "low"},]# Novel rule systems for testing rule-based learningJOL_RULE_SYSTEMS = [    {        "rule_name": "Zaplang Number System",        "rules": [            "In Zaplang, 'ko' means 1, 'bo' means 2, 'mo' means 3",            "Adding '-ra' multiplies by 10 (e.g., 'ko-ra' = 10)",            "Adding '-fi' adds 5 (e.g., 'bo-fi' = 7)",            "Numbers combine left-to-right: 'mo-ra bo-fi' = 37",        ],        "test_questions": [            {"q": "What is 'bo-ra ko-fi' in Zaplang?", "a": "26"},            {"q": "What is 'ko-ra mo' in Zaplang?", "a": "13"},            {"q": "How do you say 15 in Zaplang?", "a": "ko-ra bo-fi"},        ],        "difficulty": 2,    },    {        "rule_name": "Gridwalker Movement",        "rules": [            "Start at position (0,0) on a grid",            "'vex' moves +1 in x, 'nux' moves +1 in y",            "'rev' reverses the direction of the NEXT command only",            "'dub' doubles the distance of the NEXT command only",        ],        "test_questions": [            {"q": "After 'vex nux vex', what is the position?", "a": "(2, 1)"},            {"q": "After 'vex rev nux vex', what is the position?", "a": "(2, -1)"},            {"q": "After 'dub vex nux', what is the position?", "a": "(2, 1)"},        ],        "difficulty": 3,    },]# Distractor questions (unrelated, to create temporal distance)DISTRACTOR_QUESTIONS = [    "Name three countries in South America.",    "What is the square root of 144?",    "List the primary colors of light.",    "What is the chemical formula for glucose?",    "Name the four cardinal directions.",    "What is 17 times 13?",    "List three types of cloud formations.",    "What planet has the most moons?",]

In [ ]:
"""JOL Sub-metric Tasks: Individual leaderboard entries for each JOL metric.Splits the composite JOL benchmark into separate tasks for:- jol_gamma: Goodman-Kruskal gamma correlation (monitoring accuracy)- jol_ece: Expected Calibration Error (inverted: 1 - ECE)- jol_recall: Recall rate (in-context learning effectiveness)"""import kaggle_benchmarks as kbenchfrom dataclasses import dataclassimport numpy as npimport reimport jsonimport random# JOL data defined above@dataclassclass JOLRating:    confidence: int    reasoning: str@dataclassclass RecallAttempt:    definition: str    confidence: int@dataclassclass RuleAnswer:    answer: str    reasoning: strdef normalize(text):    text = text.lower().strip()    text = re.sub(r'[^\w\s]', ' ', text)    text = re.sub(r'\s+', ' ', text).strip()    return textdef recall_match(recalled, original, threshold=0.5):    stop_words = {"a","an","the","of","in","on","at","to","for","is",                  "that","which","and","or","but","with","by","from"}    orig_words = set(normalize(original).split()) - stop_words    recall_words = set(normalize(recalled).split()) - stop_words    if not orig_words:        return True    return len(orig_words & recall_words) / len(orig_words) >= thresholddef goodman_kruskal_gamma(x, y):    n = len(x)    concordant = discordant = 0    for i in range(n):        for j in range(i+1, n):            product = (x[i]-x[j]) * (y[i]-y[j])            if product > 0: concordant += 1            elif product < 0: discordant += 1    denom = concordant + discordant    return (concordant - discordant) / denom if denom else 0.0def compute_ece(confidences, accuracies, n_bins=5):    conf = np.array(confidences) / 100.0    acc = np.array(accuracies, dtype=float)    boundaries = np.linspace(0, 1, n_bins + 1)    ece = 0.0    total = len(conf)    for i in range(n_bins):        lo, hi = boundaries[i], boundaries[i+1]        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)        if mask.sum() == 0: continue        ece += (mask.sum() / total) * abs(acc[mask].mean() - conf[mask].mean())    return round(float(ece), 4)def _collect_jol_data(llm):    """Run the full JOL Study→JOL→Distract→Test protocol. Returns (jol_ratings, accuracies)."""    all_jol = []    all_acc = []    # Word pairs    with kbench.chats.new("jol_sub_study"):        study_prompt = "I'm going to teach you some new vocabulary words. Study each one carefully.\n\n"        for i, pair in enumerate(JOL_WORD_PAIRS):            study_prompt += f"{i+1}. **{pair['word']}**: {pair['definition']}\n"        study_prompt += "\nPlease confirm you've studied these words by saying 'Ready'."        llm.prompt(study_prompt)        for i, pair in enumerate(JOL_WORD_PAIRS):            jol_prompt = (                f"For the word **{pair['word']}** (which you just studied), "                f"rate your confidence (0-100) that you will be able to recall "                f"its definition later after some unrelated questions.\n\n"                f'Respond: {{"confidence": <0-100>, "reasoning": "<brief>"}}'            )            try:                jol = llm.prompt(jol_prompt, schema=JOLRating)                conf = max(0, min(100, jol.confidence))            except Exception:                conf = 50            all_jol.append(conf)        distractors = random.sample(DISTRACTOR_QUESTIONS, min(5, len(DISTRACTOR_QUESTIONS)))        for dq in distractors:            llm.prompt(dq)        for i, pair in enumerate(JOL_WORD_PAIRS):            recall_prompt = (                f"Earlier, I taught you the word **{pair['word']}**. "                f"What was its definition?\n\n"                f'Respond: {{"definition": "<recalled definition>", "confidence": <0-100>}}'            )            try:                r = llm.prompt(recall_prompt, schema=RecallAttempt)                recalled = r.definition            except Exception:                raw = llm.prompt(recall_prompt)                try:                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                    recalled = str(parsed.get("definition", raw))                except Exception:                    recalled = raw            all_acc.append(recall_match(recalled, pair["definition"]))    # Rule systems    for rs in JOL_RULE_SYSTEMS:        with kbench.chats.new(f"jol_sub_rule_{rs['rule_name']}"):            rules_text = f"Learn the following rule system: **{rs['rule_name']}**\n\n"            for r in rs["rules"]:                rules_text += f"- {r}\n"            rules_text += "\nSay 'Ready' when you've studied these rules."            llm.prompt(rules_text)            jol_prompt = (                f"Rate your confidence (0-100) that you can correctly apply "                f"the {rs['rule_name']} rules to answer test questions.\n\n"                f'Respond: {{"confidence": <0-100>, "reasoning": "<brief>"}}'            )            try:                jol = llm.prompt(jol_prompt, schema=JOLRating)                conf = max(0, min(100, jol.confidence))            except Exception:                conf = 50            llm.prompt(random.choice(DISTRACTOR_QUESTIONS))            rule_correct = 0            for tq in rs["test_questions"]:                test_prompt = (                    f"Using the {rs['rule_name']} rules you learned, answer:\n{tq['q']}\n\n"                    f'Respond: {{"answer": "<answer>", "reasoning": "<steps>"}}'                )                try:                    ans = llm.prompt(test_prompt, schema=RuleAnswer)                    answer = ans.answer                except Exception:                    raw = llm.prompt(test_prompt)                    try:                        parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                        answer = str(parsed.get("answer", raw))                    except Exception:                        answer = raw                if normalize(tq["a"]) in normalize(answer):                    rule_correct += 1            all_jol.append(conf)            all_acc.append(rule_correct / len(rs["test_questions"]) >= 0.5)    return all_jol, all_acc@kbench.task(name="metacog_jol_gamma")def metacog_jol_gamma(llm) -> float:    """JOL Gamma — ordinal association between JOL ratings and recall. Normalized to [0,1]. Human range: 0.70–0.95."""    jols, accs = _collect_jol_data(llm)    gamma = goodman_kruskal_gamma(jols, [int(a) for a in accs])    return round((gamma + 1) / 2, 4)@kbench.task(name="metacog_jol_ece")def metacog_jol_ece(llm) -> float:    """JOL Calibration (1 - ECE) — how well JOL ratings match actual recall."""    jols, accs = _collect_jol_data(llm)    ece = compute_ece(jols, accs)    return round(1 - ece, 4)@kbench.task(name="metacog_jol_recall")def metacog_jol_recall(llm) -> float:    """JOL Recall Rate — proportion of items successfully recalled (in-context learning)."""    jols, accs = _collect_jol_data(llm)    return round(sum(accs) / len(accs), 4)metacog_jol_gamma.run(llm=kbench.llm)metacog_jol_ece.run(llm=kbench.llm)metacog_jol_recall.run(llm=kbench.llm)